## Example: Properties of a Three-State Hidden Markov Model
In this example, you will build and simulate a Hidden Markov Model (HMM) that separates hidden internal states from observable outputs, demonstrating how latent processes can generate observed data.

> __Learning Objectives:__
>
> After completing this activity, students will be able to:
> * **Construct hidden Markov models with two-layer structure:** We build a three-state HMM with hidden states representing moods (happy, neutral, sad) governed by a transition matrix and observable outputs represented by emojis controlled by an emission probability matrix.
> * **Configure emission probability matrices:** We construct the emission probability matrix that maps hidden states to observable outputs, allowing for probabilistic mismatches between internal states and external observations (e.g., 90% correct observation with 10% noise).
> * **Simulate HMM dynamics and validate output distributions:** We implement the forward sampling algorithm to generate sequences of hidden states and observable outputs over 50,000 time steps and verify that the observed output frequencies match the expected probabilities derived from the stationary distribution and emission matrix.

This example demonstrates how HMMs model systems where internal states are hidden but generate observable signals. Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Implementation
Before we start this example, let's set up the `compute_stationary_distribution(...)` method and specify some constants. We'll use the `compute_stationary_distribution(...)` method to compute the stationary distribution $\pi$.
```julia
compute_stationary_distribution(P::Array{Float64,2}; 
        maxcount::Int = 100, ϵ::Float64 = 0.1) -> Array{Float64,2}
```
> Iteratively computes a stationary distribution. Computation stops if $\|P_{\text{new}} - P_{\text{prev}}\|_{F} < \epsilon$ or the max number of iterations is hit. 

In [2]:
function compute_stationary_distribution(P::Array{Float64,2}; 
        maxcount::Int = 100, ϵ::Float64 = 0.1)::Array{Float64,2}

    # initialize -
    counter = 1; # initialize the iteration counter
    is_ok_to_stop = false; # flag for while loop
    P_prev = P; # initialize P_prev to P (this is P^1)
    P_new = nothing; # initialize P_new matrix
    
    # main loop - iterate until the difference ||P_new - P_prev|| <= ϵ -or- we run out of iterations
    while (is_ok_to_stop == false)
        P_new = P_prev * P; # compute new P matrix: P_prev holds P^counter, so P_new = P^(counter+1)
        if (norm(P_new - P_prev) <= ϵ || counter >= maxcount)
            is_ok_to_stop = true;
        end
        P_prev = P_new; # update P_prev for the next iteration
        counter += 1; # update the counter
    end

    # return -
    return P_new;
end;

### Constants 
In the simulations below, we'll need some constant values that we set here. In particular, we set a value for the `number_of_hidden_states` variable, the `number_of_simulation_steps` variable (the number of steps that we take in a Markov chain), and the `number_of_observable_states` variable:

In [3]:
number_of_hidden_states = 3; # how many hidden states do we have?
number_of_observable_states = 3; # how many observable states do we have?
number_of_simulation_steps = 50000; # number of simulation steps

___

## What makes this model "hidden"?
In the previous Markov model notebook, we directly observed which state the system was in at each time step. In a Hidden Markov Model, the true state (mood: happy, neutral, sad) is **hidden**: we cannot observe it directly. Instead, we only see **observable outputs** (emojis) that give us noisy information about the hidden state.

The key components of an HMM are:
* **Hidden states** $s \in \mathcal{S}$: The true underlying state (mood) governed by transition matrix $\mathbf{P}$
* **Observable outputs** $o \in \mathcal{O}$: What we actually see (emojis) governed by emission matrix $\mathbf{E}$
* **Emission matrix** $\mathbf{E}$: Maps hidden states to observable outputs with some noise/uncertainty

For example, if someone is truly happy (hidden state), they will usually show 😄 (90% of the time), but occasionally might show 😐 or 😞 (10% of the time) due to social masking or measurement error.
___

<div>
    <center>
        <img src="figs/Fig-ThreeState-HMM-Schematic.svg" width="580"/>
    </center>
</div>

## Task 1: Set up the transition matrix $\mathbf{P}$
In this task, we'll set up the transition matrix $\mathbf{P}$ for a three-state [Markov chain model](https://en.wikipedia.org/wiki/Markov_chain). Suppose we have three states $\mathcal{S}\equiv\left\{\text{happy},\text{neutral},\text{sad}\right\}$ and the probability of moving from state $i$ to state $j$, denoted as $p_{ij}$, is an element of the matrix $\mathbf{P} \in \mathbb{R}^{3\times{3}}$.

In [4]:
P = [
    0.05 0.95 0.0 ; # moves for state 1 = happy
    0.6 0.2 0.2 ; # moves for state 2 = neutral
    0.0 0.3 0.7 ; # moves for state 3 = sad
];

### Check: Do the rows of the transition matrix $\mathbf{P}$ sum to `1`?
We know that the rows of the transition matrix $\mathbf{P}$ must sum to `1`, i.e., if we are in state $s_{i}\in\mathcal{S}$ at time $t$, then at time $t+1$ we must be in some state $s_{j}\in\mathcal{S}$. 

> __Check:__ Let's verify that the transition matrix $\mathbf{P}$ meets this criterion using the [@assert macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) by iterating over the rows of the transition matrix $\mathbf{P}$ and checking the sum of each row. If any row does not meet this criterion, an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) will be thrown.

So, what do we see?

In [5]:
let
    for i ∈ 1:number_of_hidden_states
        @assert sum(P[i,:]) == 1
    end
end;

Now that we are sure that the transition matrix $\mathbf{P}$ is proper, we populate the `hidden_state_probability_dictionary::Dict{Int,Categorical}`, which holds the [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) modeling the transition probability for each hidden state $s\in\mathcal{S}$, i.e., the probability that we transition from state $i\rightarrow{j}$ in the next time step:

In [6]:
hidden_state_probability_dictionary = let

    # initialize -
    hidden_state_probability_dictionary = Dict{Int,Categorical}();
    
    # main loop -
    for i ∈ 1:number_of_hidden_states
        hidden_state_probability_dictionary[i] = Categorical(P[i,:]) # create a categorical distribution for each hidden state
    end

    hidden_state_probability_dictionary # return
end;

Let's test the hidden state transitions by starting from state 2 (neutral) and running 10 trials to see how the system moves between states according to the transition probabilities in $\mathbf{P}$:

In [7]:
let

    # initialize -
    current_hidden_state = 2; # current state that we are in *now*
    number_of_trials = 10; # how many trials to run

    for i ∈ 1:number_of_trials
        next_hidden_state = rand(hidden_state_probability_dictionary[current_hidden_state]) # sample the next state from the categorical distribution
        println("Trial $(i): current hidden state = $(current_hidden_state), next hidden state = $(next_hidden_state)")
        current_hidden_state = next_hidden_state; # update the current hidden state
    end
end;

Trial 1: current hidden state = 2, next hidden state = 1


Trial 2: current hidden state = 1, next hidden state = 2
Trial 3: current hidden state = 2, next hidden state = 1
Trial 4: current hidden state = 1, next hidden state = 2
Trial 5: current hidden state = 2, next hidden state = 1
Trial 6: current hidden state = 1, next hidden state = 2
Trial 7: current hidden state = 2, next hidden state = 1
Trial 8: current hidden state = 1, next hidden state = 2
Trial 9: current hidden state = 2, next hidden state = 1
Trial 10: current hidden state = 1, next hidden state = 2


___

## Task 2: Compute the stationary distribution $\pi$
In this task, we'll compute the stationary distribution $\pi$ for our example [Markov chain](https://en.wikipedia.org/wiki/Markov_chain) using the `compute_stationary_distribution(...)` method defined above. This version of the method uses a `while-loop`; we iterate until the difference $\|\mathbf{P}^{i+1} - \mathbf{P}^{i}\|_{F} < \epsilon$, or we run out of iterations (the iteration counter exceeds the `maxcount` argument).

In [8]:
π̄ = compute_stationary_distribution(P, ϵ = 1e-9, maxcount = 10000) # iteratively compute the stationary distribution

3×3 Matrix{Float64}:
 0.274809  0.435115  0.290076
 0.274809  0.435115  0.290076
 0.274809  0.435115  0.290076

Finally, create a [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) using the stationary probability of our Markov chain using the [Distributions.jl](https://github.com/JuliaStats/Distributions.jl) package, save this distribution in the variable `d`:

In [9]:
d = Categorical(π̄[1,:]); # steady-state stationary distribution

___

## Task 3: Set up the emission probability matrix $\mathbf{E}$
In this task, we set up the emission probability matrix $\mathbf{E}$, which links the hidden and observable [Markov chain](https://en.wikipedia.org/wiki/Markov_chain) states.
Now that we have the stationary distribution for the hidden layer of our [Hidden Markov Model](https://en.wikipedia.org/wiki/Hidden_Markov_model), let's set up the emission probability matrix $\mathbf{E}$:

In [10]:
E = [
    0.90 0.05 0.05 ; # 1 happy (but sometimes we see other faces)
    0.05 0.90 0.05 ; # 2 neutral (but sometimes we see other faces)
    0.05 0.05 0.90 ; # 3 sad (but sometimes we see other faces)
];
# E = diagm(ones(3)) # we never have a missed guess ...

Populate the `emission_probability_dict::Dict{Int,Categorical}`, which holds the [categorical distribution](https://en.wikipedia.org/wiki/Categorical_distribution) modeling the emission probability for each hidden state $s\in\mathcal{S}$, i.e., the probability of what output $o_{i}\in\mathcal{O}$ we expect to see if we are in $s\in\mathcal{S}$:

In [11]:
emission_probability_dict = let
    emission_probability_dict = Dict{Int,Categorical}()
    for i ∈ 1:number_of_hidden_states
        emission_probability_dict[i] = Categorical(E[i,:])
    end
    emission_probability_dict
end;

### Simulate the output from the HMM
In this task, we simulate the evolution of the hidden Markov model.
Let's implement the pseudo-code from the lecture, where each observable state corresponds to an [Emoji](https://en.wikipedia.org/wiki/Emoji). We store this relationship in the `observable_emoji_map` variable, which is a dictionary with keys corresponding to observable states $o\in\mathcal{O}$ and [Emoji](https://en.wikipedia.org/wiki/Emoji) values.

In [12]:
observable_emoji_map = Dict{Int,Any}();
observable_emoji_map[1] = `😄`;
observable_emoji_map[2] = `😐`;
observable_emoji_map[3] = `😞`;

__Simulation algorithm:__ 

For `number_of_simulation_steps`, starting from some initial state $s\in\mathcal{S}$: 
* First we get the hidden state distribution from the `hidden_state_probability_dictionary`, we then generate a new state $s^{\prime}$, access the emission distribution from the `emission_probability_dict` that corresponds to $s^{\prime}$, and we generate a random observable output $o_{i}$.
* Next, we save both the hidden state $s^{\prime}$ and the output $o_{i}$ for this iteration in the `hidden_simulation_dict` and `output_simulation_dict` variables, respectively.
* Finally, we update the current state $s_{i}\leftarrow{s}^{\prime}$ and move onto the next iteration.

These are stored in `hidden_simulation_dict::Dict{Int,Int}` and `output_simulation_dict::Dict{Int,Any}`, used in later cells.

In [13]:
(hidden_simulation_dict, output_simulation_dict) = let
    output_simulation_dict = Dict{Int,Any}()
    hidden_simulation_dict = Dict{Int,Int}();
    sᵢ = 1;
    for i ∈ 1:number_of_simulation_steps

        # get the categorical distribution for sᵢ 
        dᵢ = hidden_state_probability_dictionary[sᵢ];
        
        # compute the *next* hidden state -
        s′ = rand(dᵢ);

        # next, compute what output we see from this state -
        oᵢ = emission_probability_dict[s′] |> o -> rand(o);

        # capture -
        hidden_simulation_dict[i] = s′
        output_simulation_dict[i] = observable_emoji_map[oᵢ]

        # update -
        sᵢ = s′;
    end

    (hidden_simulation_dict, output_simulation_dict)
end;

Let's print the hidden state and observed emoji for the first 20 simulated time steps:

In [14]:
foreach(i->println("$(hidden_simulation_dict[i]),$(output_simulation_dict[i])"), 1:20); # print first 20 time steps

2,`😐`
1,`😄`
2,`😐`
1,`😄`
2,`😐`
2,`😐`
3,`😞`
2,`😐`
1,`😄`
2,`😐`
1,`😄`
2,`😐`
1,`😄`
2,`😐`
1,`😄`
2,`😐`
2,`😞`
1,`😄`
2,`😐`
3,`😞`


### Validate simulation results against theoretical predictions
Now let's validate our simulation by comparing the observed emoji frequencies to the theoretical predictions. The theoretical probability of observing emoji $o_j$ is:

$$
P(o_j) = \sum_{i=1}^{3} \pi_i \times E_{ij}
$$

where $\pi_i$ is the stationary probability of hidden state $i$ and $E_{ij}$ is the emission probability from hidden state $i$ to observable output $j$.

In [15]:
let
    # Count observed emoji frequencies
    emoji_counts = Dict{Any,Int}(
        `😄` => 0,
        `😐` => 0,
        `😞` => 0
    );
    
    for (key, value) ∈ output_simulation_dict
        emoji_counts[value] += 1
    end
    
    # Compute observed frequencies
    observed_freq = Dict{Any,Float64}(
        emoji => count/number_of_simulation_steps 
        for (emoji, count) ∈ emoji_counts
    );
    
    # Compute theoretical predictions: P(emoji_j) = Σᵢ π̄[i] × E[i,j]
    theoretical_prob = zeros(number_of_observable_states);
    for j ∈ 1:number_of_observable_states  # for each observable emoji
        for i ∈ 1:number_of_hidden_states  # sum over all hidden states
            theoretical_prob[j] += π̄[1,i] * E[i,j]
        end
    end
    
    # Create comparison table
    df = DataFrame(
        Emoji = [`😄`, `😐`, `😞`],
        Observed_Count = [emoji_counts[`😄`], emoji_counts[`😐`], emoji_counts[`😞`]],
        Observed_Frequency = [observed_freq[`😄`], observed_freq[`😐`], observed_freq[`😞`]],
        Theoretical_Probability = theoretical_prob,
        Difference = [abs(observed_freq[`😄`] - theoretical_prob[1]), 
                     abs(observed_freq[`😐`] - theoretical_prob[2]), 
                     abs(observed_freq[`😞`] - theoretical_prob[3])]
    );
    
    pretty_table(df, backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end;

 ----------------- ---------------- -------------------- -----------------------
            Emoji   Observed_Count   Observed_Frequency   Theoretical_Probabil ⋯
              Cmd            Int64              Float64                   Floa ⋯
 ----------------- ---------------- -------------------- -----------------------
  `\e😄\e`            14173              0.28346                  0.283 ⋯
  `\e😐\e`            21030               0.4206                  0.419 ⋯
  `\e😞\e`            14797              0.29594                  0.296 ⋯
 ----------------- ---------------- -------------------- -----------------------
                                                               2 columns omitted


___

## Summary
In this example, we constructed a three-state Hidden Markov Model with hidden mood states and observable emoji outputs, simulated its dynamics over 50,000 time steps, and validated that observed output frequencies matched theoretical predictions.

> __Key Takeaways:__
> 
> * **Two-layer HMM architecture:** We built an HMM with a hidden layer governed by a 3×3 transition matrix P (representing mood transitions) and an observable layer controlled by a 3×3 emission matrix E that maps hidden states to emoji outputs with 90% accuracy and 10% observation noise.
> * **Forward sampling algorithm implementation:** We simulated the HMM by iteratively sampling the next hidden state from the transition probability distribution, then sampling the observable output from the emission probability distribution conditioned on the current hidden state, generating synchronized sequences of both hidden and observable states.
> * **Empirical validation of output probabilities:** We computed the frequency of each emoji in the 50,000-step simulation and confirmed that the observed probabilities converged to values consistent with the product of the stationary distribution π and the emission probabilities, demonstrating that HMM simulations produce statistically consistent outputs.

Hidden Markov Models provide a framework for modeling systems where an unobservable process generates observable signals, as demonstrated here with the mood/emoji example.
___